# Chapter 17 &mdash; Variable Ordering Matters: the Magnitude Comparator

**Concept 3 of the Chapter 17 decomposition:** *Variable Ordering Matters: the Magnitude Comparator*

Interleaving $x_2,y_2,x_1,y_1,x_0,y_0$ lets the comparator decide early; a bad order forces it to remember everything.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter17/Concept-Variable-Ordering-Matters/Concept-Variable-Ordering-Matters.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A BDD's size depends drastically on the **variable order**, and the magnitude
comparator $X > Y$ is the standard illustration.

* **Interleaved** $x_2,y_2,x_1,y_1,x_0,y_0$: at each bit position the machine compares
  two bits and can **decide immediately** if they differ. It only needs to remember
  "still equal so far" &mdash; a constant amount. The BDD is **linear** in $n$.
* **Separated** $x_2,x_1,x_0,y_2,y_1,y_0$: all of $X$ is read before any of $Y$, so
  the machine must remember **which of $2^n$ values** $X$ took. The BDD is
  **exponential**.

Note this is the Chapter 5 lesson again: *state names as compressed history*. A good
order gives a small residual-function count; a bad one gives $2^n$.

## 2. Definitions

### The BDD package

In [ ]:
# --- a minimal BDD package ----------------------------------------------
# A node is either the terminal 0/1, or ('n', var_index, low, high) where
# low is the 0-branch and high the 1-branch.  Hash consing (the `unique`
# table) is what makes the representation canonical: structurally equal
# subgraphs become the SAME Python object, so equality is pointer equality.
ZERO, ONE = 0, 1

class BDD:
    def __init__(self, nvars):
        self.nvars = nvars
        self.unique = {}          # (var, low, high) -> node  -- hash consing
        self.apply_cache = {}

    def mk(self, var, low, high):
        if low is high: return low            # REDUCTION 1: skip a useless test
        key = (var, id(low), id(high), self._k(low), self._k(high))
        if key in self.unique: return self.unique[key]   # REDUCTION 2: share
        node = ('n', var, low, high)
        self.unique[key] = node
        return node

    def _k(self, n):
        return n if n in (ZERO, ONE) else ('n', n[1], self._k(n[2]), self._k(n[3]))

    def var(self, i):
        return self.mk(i, ZERO, ONE)

    def apply(self, op, a, b):
        key = (op, self._k(a), self._k(b))
        if key in self.apply_cache: return self.apply_cache[key]
        if a in (ZERO, ONE) and b in (ZERO, ONE):
            r = ONE if op(bool(a), bool(b)) else ZERO
        else:
            va = a[1] if a not in (ZERO, ONE) else self.nvars
            vb = b[1] if b not in (ZERO, ONE) else self.nvars
            v = min(va, vb)
            al, ah = (a[2], a[3]) if va == v else (a, a)
            bl, bh = (b[2], b[3]) if vb == v else (b, b)
            r = self.mk(v, self.apply(op, al, bl), self.apply(op, ah, bh))
        self.apply_cache[key] = r
        return r

    def NOT(self, a):  return self.apply(lambda x, y: not x, a, a)
    def AND(self, a, b): return self.apply(lambda x, y: x and y, a, b)
    def OR(self, a, b):  return self.apply(lambda x, y: x or y, a, b)
    def XOR(self, a, b): return self.apply(lambda x, y: x != y, a, b)

    def evaluate(self, node, assign):
        while node not in (ZERO, ONE):
            node = node[3] if assign[node[1]] else node[2]
        return bool(node)

    def size(self, node):
        seen = set()
        def walk(n):
            if n in (ZERO, ONE): return
            k = self._k(n)
            if k in seen: return
            seen.add(k); walk(n[2]); walk(n[3])
        walk(node)
        return len(seen)

    def onset(self, node, order=None):
        from itertools import product
        out = []
        for bits in product([False, True], repeat=self.nvars):
            a = {i: bits[i] for i in range(self.nvars)}
            if self.evaluate(node, a):
                out.append(''.join('1' if bits[i] else '0' for i in range(self.nvars)))
        return sorted(out)

### The comparator, under two orders

In [ ]:
def comparator_bdd(n, interleaved=True):
    b = BDD(2 * n)
    if interleaved:
        xi = [2 * k for k in range(n)]          # x_{n-1}, y_{n-1}, x_{n-2}, ...
        yi = [2 * k + 1 for k in range(n)]
    else:
        xi = list(range(n))                      # all of X, then all of Y
        yi = list(range(n, 2 * n))
    x = [b.var(i) for i in xi]
    y = [b.var(i) for i in yi]
    # X > Y, most significant bit first
    gt, eq = ZERO, ONE
    for k in range(n):
        gt = b.OR(gt, b.AND(eq, b.AND(x[k], b.NOT(y[k]))))
        eq = b.AND(eq, b.NOT(b.XOR(x[k], y[k])))
    return b, gt

## 3. Tests

The comparator is correct under both orders.

In [ ]:
from itertools import product
def check(n, interleaved):
    b, g = comparator_bdd(n, interleaved)
    for bits in product([0, 1], repeat=2 * n):
        a = {i: bool(bits[i]) for i in range(2 * n)}
        if interleaved:
            X = int(''.join(str(bits[2 * k]) for k in range(n)), 2)
            Y = int(''.join(str(bits[2 * k + 1]) for k in range(n)), 2)
        else:
            X = int(''.join(str(bits[k]) for k in range(n)), 2)
            Y = int(''.join(str(bits[n + k]) for k in range(n)), 2)
        assert b.evaluate(g, a) == (X > Y), (bits, X, Y)
    return b.size(g)

for n in [1, 2, 3]:
    print("  n=%d : interleaved %2d nodes, separated %2d nodes"
          % (n, check(n, True), check(n, False)))

**The gap grows.** Interleaved is linear; separated is exponential.

In [ ]:
print("%-5s %-14s %-14s %s" % ("n", "interleaved", "separated", "ratio"))
for n in [1, 2, 3, 4]:
    b1, g1 = comparator_bdd(n, True)
    b2, g2 = comparator_bdd(n, False)
    s1, s2 = b1.size(g1), b2.size(g2)
    print("%-5d %-14d %-14d %.1fx" % (n, s1, s2, s2 / float(s1)))
b1, g1 = comparator_bdd(4, True); b2, g2 = comparator_bdd(4, False)
assert b2.size(g2) > b1.size(g1)

**Why:** the separated order must remember all of $X$.

In [ ]:
print("interleaved : read x_k and y_k together")
print("              -> only two residual functions matter:")
print("                 'still equal' and 'already decided'")
print()
print("separated   : read all of X, then all of Y")
print("              -> must distinguish all 2^n values of X,")
print("                 because each leads to a different residual on Y")

Counting the residual functions directly confirms it.

In [ ]:
def residuals_after_prefix(n, k, interleaved):
    # how many distinct functions of the REMAINING variables?
    seen = set()
    for pre in product([0, 1], repeat=k):
        rows = []
        for rest in product([0, 1], repeat=2 * n - k):
            bits = pre + rest
            if interleaved:
                X = int(''.join(str(bits[2 * j]) for j in range(n)), 2)
                Y = int(''.join(str(bits[2 * j + 1]) for j in range(n)), 2)
            else:
                X = int(''.join(str(bits[j]) for j in range(n)), 2)
                Y = int(''.join(str(bits[n + j]) for j in range(n)), 2)
            rows.append(X > Y)
        seen.add(tuple(rows))
    return len(seen)

n = 3
for k in [2, 4]:
    print("  after %d variables : interleaved %d residuals, separated %d"
          % (k, residuals_after_prefix(n, k, True),
             residuals_after_prefix(n, k, False)))
print("\nresiduals after reading all of X (k=%d, separated) : %d = 2^%d"
      % (n, residuals_after_prefix(n, n, False), n))

Same lesson as Chapter 5: the state **is** compressed history.

In [ ]:
print("Chapter 5, Concept 6 : state = longest useful suffix")
print("Chapter 5, Concept 10: look-back forces 2^N states")
print("here                 : variable order decides WHAT must be remembered")
print()
print("A good order lets you forget early.  That is the whole art.")

## 4. Exercises


1. Find a third order for the 3-bit comparator. Where does it land?
2. Give a function whose BDD is small under **every** order.
3. Give one that is exponential under every order. (Hint: multiplication.)

In [ ]:
# Your work for the exercises above.